# M67 — exact A2 Core ML feasibility export

Revision: `M67-A2-EXPORT-RUNTIME-AUDIT-2026-10-06-r1`. This is a bounded, zero-training export check for the **original A2 epoch-130 checkpoint** (MobileNetV4 Conv Medium + full MonoDETR). It answers whether the exact trained graph can be exported and preserve its raw outputs; it does **not** measure iPhone speed.

Use a Colab **GPU runtime with CUDA 13.0**; A100 is recommended, L4 is acceptable. Run sections 1–5 in order. Required on Drive: the completed M66 manifest, unchanged A2 epoch-130 checkpoint, and exact Chen validation split. Only the first 16 fixed validation images are restored. Training, optimizer updates, quantization, checkpoint promotion, AP evaluation and deployment are not performed.

The deliverable is `m67_a2_export_results.zip`, containing provenance, fixed inputs, FP32 Core ML package, parity reports and durable logs. If export and fixed-input parity pass, return the ZIP for review. **Do not connect the phone yet**: Mac Core ML parity is the next gate. No app is created or changed by M67. Before device testing, we will reuse the existing benchmark harness or the actual product-app source if available; we will not create another app.

## 1. Mount Drive, pin the reviewed inputs, and load the current project scripts

The manifest path below comes from the completed M66 run. Do not change it to a control/KD checkpoint. If the M66 output folder differs, edit only `M66_MANIFEST`. The script rejects Markdown-formatted Git URLs and never resets or deletes an existing checkout.

In [ ]:
# M67-A2-EXPORT-RUNTIME-AUDIT-2026-10-06-r1
from pathlib import Path
from collections import deque
import hashlib, json, os, runpy, shlex, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M67-A2-EXPORT-RUNTIME-AUDIT-2026-10-06-r1'
RUN_ID = 'm67_a2_fp32_export_r1'  # Use a new ID only for an intentionally fresh run.
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M67_A2')
A2_URL = 'https://github.com/ZrrSkywalker/MonoDETR.git'
A2_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
M66_MANIFEST = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m66_r0_a2_feature/m66_r0_a2_vehicle_feature_r1/m66_manifest.json')
M66_SIGNATURE = '68c2247a9ab28603e10f7f9748fc17a43e5bdefeedc57eebe87e2c62f430d968'
A2_SHA = 'ed2134a98acbf1ab2fc61f7c8749b38fdfd2418e7f7932593e5e37a8d9ef33f4'
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET = Path('/content/kitti_m67')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_m67_export') / RUN_ID
PYTHON = Path(f'/content/{RUN_ID}_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
SCRIPT = MOBILE_REPO/'scripts/audit_m67_a2_coreml.py'
RUNTIME_RECEIPT = OUTPUT/'m67_runtime_receipt.json'
MANIFEST = OUTPUT/'m67_manifest.json'
SETUP_READY = MANIFEST_READY = EXPORT_READY = False
ENV = os.environ.copy()

if not M66_MANIFEST.is_file(): raise FileNotFoundError(f'Missing reviewed M66 manifest: {M66_MANIFEST}')
m66 = json.loads(M66_MANIFEST.read_text())
if m66.get('manifest_sha256') != M66_SIGNATURE: raise RuntimeError('M66 manifest is not the reviewed run')
body = {k:v for k,v in m66.items() if k != 'manifest_sha256'}
if hashlib.sha256(json.dumps(body,sort_keys=True,allow_nan=False).encode()).hexdigest() != M66_SIGNATURE: raise RuntimeError('M66 manifest self-signature failed')
if m66['models']['a2'].get('checkpoint_sha256') != A2_SHA: raise RuntimeError('M66 manifest points to another A2 checkpoint')
A2_CHECKPOINT = Path(m66['models']['a2']['checkpoint'])
if not A2_CHECKPOINT.is_file(): raise FileNotFoundError(f'Original A2 epoch130 checkpoint must remain on Drive: {A2_CHECKPOINT}')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
if sha256(A2_CHECKPOINT) != A2_SHA: raise RuntimeError('Original A2 checkpoint bytes differ')
if not SPLITS.is_dir(): raise FileNotFoundError(SPLITS)
OUTPUT.mkdir(parents=True,exist_ok=True)
print(REVISION, '\nA2 epoch130:', A2_CHECKPOINT, '\nOutput:', OUTPUT, flush=True)

def run_logged(command,name,cwd=None,allow_failure=False):
    command=[str(x) for x in command]
    log_path=OUTPUT/'colab_logs'/f'{name}.log'; log_path.parent.mkdir(parents=True,exist_ok=True)
    tail=deque(maxlen=60)
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n'); log.flush()
        with subprocess.Popen(command,cwd=str(cwd) if cwd else None,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout: print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            rc=p.wait()
        log.write(f'\nEXIT CODE: {rc}\n'); log.flush()
    if rc and not allow_failure: raise RuntimeError(f'Exit {rc}; full log={log_path}\n'+'\n'.join(tail))
    return rc

def checkout(path,url,commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'): raise ValueError('Use a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git','clone','--no-checkout',url,path],f'clone_{path.name}')
        run_logged(['git','checkout','--detach',commit],f'pin_{path.name}',path)
    if not (path/'.git').is_dir(): raise RuntimeError(f'Preserve {path}; choose a fresh checkout path')
    head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin=subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip().rstrip('/')
    if head != commit or origin != url: raise RuntimeError(f'Unexpected source identity at {path}; do not reset it')

if not MOBILE_REPO.exists(): run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
if not SCRIPT.is_file(): raise FileNotFoundError('M67 script is not on main; reload the published notebook after it is merged')
ns=runpy.run_path(str(SCRIPT),run_name='m67_revision_check')
if ns['REVISION'] != REVISION: raise RuntimeError('Notebook/script revisions differ; reload the latest main notebook')
print('Section 1 ready: exact M66/A2 identities and current main script confirmed.')


## 2. Build a fresh CUDA 13 runtime and pinned A2 source

This reuses the established M64 CUDA 13 environment and A2 deformable-attention build. Build outputs stay inside a new `MonoDETR_M67_A2` checkout. The Core ML export patch is added only after the native attention binary is built. If CUDA 13 or the GPU driver is unavailable, stop and return the log; do not install another Torch/CUDA wheel into Colab's base environment.

In [ ]:
SETUP_READY = MANIFEST_READY = EXPORT_READY = False
if not SCRIPT.is_file(): raise RuntimeError('Run section 1 first')
if not (CUDA/'bin/nvcc').is_file(): raise FileNotFoundError(f'CUDA 13 nvcc not found: {CUDA}')
helpers=runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='m67_cuda13_helpers')
ENV=helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m67_runtime.py','--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',RUNTIME_RECEIPT],'m67_runtime',MOBILE_REPO)
checkout(A2_REPO,A2_URL,A2_COMMIT)
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'m67_patch_build_sources',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py','--repo',A2_REPO,'--role','a2'],'m67_build_attention',MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_monodetr_coreml_export.py','--monodetr-repo',A2_REPO],'m67_patch_coreml_export',MOBILE_REPO)
SETUP_READY=True
print('Section 2 ready. Exact A2 weights are still unchanged; zero training updates.')


## 3. Restore only 16 Chen-val examples and freeze the audit manifest

The script checks the exact 3,769-ID Chen validation list, then restores only its first 16 images, labels and calibration files from existing local/Drive dataset folders. It does not download or modify KITTI. Candidate paths are searched read-only; edit/add a candidate only if your existing dataset is stored elsewhere.

In [ ]:
if not SETUP_READY: raise RuntimeError('Run sections 1 and 2 first')
if not SPLITS.is_dir(): raise FileNotFoundError(SPLITS)
candidates=['/content/kitti_m66','/content/kitti_m65','/content/kitti_m64','/content/kitti_m62','/content/kitti_m61','/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti']
command=[PYTHON,SCRIPT,'prepare','--a2-manifest',M66_MANIFEST,'--repo',A2_REPO,'--output',OUTPUT,'--dataset-root',DATASET,'--split-dir',SPLITS,'--runtime-receipt',RUNTIME_RECEIPT]
for candidate in candidates: command += ['--candidate',candidate]
run_logged(command,'m67_prepare',MOBILE_REPO)
manifest=json.loads(MANIFEST.read_text())
assert manifest['checkpoint_sha256']==A2_SHA and manifest['checkpoint_epoch']==130
assert len(manifest['sample_ids'])==16 and manifest['optimizer_steps']==0
assert manifest['training_authorized'] is False and manifest['quantization_authorized'] is False
assert manifest['deployment_authorized'] is False and manifest['iphone_performance_measured'] is False
MANIFEST_READY=True
print('Section 3 ready: 16 fixed inputs frozen; original A2 checkpoint and no-training policy verified.')


## 4. Run fixed-input native/export parity and FP32 conversion

The audit checks CUDA portable-export vs native A2, CPU and TorchScript vs native outputs, then converts to an iOS 17 FP32 Core ML package. CUDA timing is recorded only as a cloud-GPU diagnostic. It is not an iPhone timing estimate. If this gate fails, preserve the log and do not proceed to Mac or phone testing. The cell captures a failed process so section 5 can still bundle its evidence.

In [ ]:
EXPORT_READY=False
if not MANIFEST_READY: raise RuntimeError('Run sections 1–3 first')
rc=run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,'audit','--manifest',MANIFEST],'m67_export_audit',MOBILE_REPO,allow_failure=True)
if rc==0:
    report=json.loads((OUTPUT/'m67_export_audit.json').read_text())
    assert report['complete'] and report['export_gate_passed'] and report['optimizer_steps']==0
    assert report['iphone_performance_measured'] is False and report['deployment_authorized'] is False
    EXPORT_READY=True
    print('Export gate passed. This still does not establish Mac or iPhone performance.')
else:
    print('Export gate failed. Preserve all files; do not change tolerances or run phone timing. Section 5 will package the failure evidence.')


## 5. Bundle artifacts for review and stop

Run this even if a prior audit failed; it includes available manifests, partial outputs, reports and durable logs. Return `m67_a2_export_results.zip`. If the export gate passed, the next separate check is macOS Core ML parity on the same 16 stored inputs. We will review that bundle and then prepare physical-device measurement using the existing benchmark harness (or the product app if its source is available). This notebook has no iPhone speed result and makes no architecture/accuracy decision.

In [ ]:
run_logged([PYTHON,SCRIPT,'bundle','--output',OUTPUT],'m67_bundle',MOBILE_REPO)
BUNDLE=OUTPUT/'m67_a2_export_results.zip'
if not BUNDLE.is_file(): raise FileNotFoundError(BUNDLE)
print('Return this file:',BUNDLE,'size_bytes=',BUNDLE.stat().st_size)
print('Export gate passed:',EXPORT_READY)
print('STOP: no Mac parity, iPhone measurement, training, quantization or deployment was run.')
